In [1]:
# If mlxtend is not installed, run this once:
# !pip install mlxtend

import pandas as pd
import numpy as np

from mlxtend.frequent_patterns import fpgrowth, association_rules

In [4]:
order_products = pd.read_csv('order_products__prior.csv') #tells which products were in each order
products = pd.read_csv('products.csv')   #it maps product_id to department_id
departments = pd.read_csv('departments.csv') 


In [5]:
# order_id + product_id -> add department_id
basket_data = pd.merge(
    order_products,
    products[['product_id', 'department_id']],
    on='product_id',
    how='left'
)

# department_id -> add department name
basket_data = pd.merge(
    basket_data,
    departments,
    on='department_id',
    how='left'
)

basket_data.head()

,order_id,product_id,add_to_cart_order,reordered,department_id,department
0,2,33120,1,1,16,dairy eggs
1,2,28985,2,1,4,produce
2,2,9327,3,0,13,pantry
3,2,45918,4,1,13,pantry
4,2,30035,5,0,13,pantry


In [6]:
# Keep only useful columns
order_departments = basket_data[['order_id', 'department']].dropna()

# Remove duplicates so one department appears only once per order so if in 1 order 5 items from dairy then count 'dairy' once for tht order
order_departments = order_departments.drop_duplicates()

order_departments.head()

,order_id,department
0,2,dairy eggs
1,2,produce
2,2,pantry
9,3,dairy eggs
11,3,produce


In [7]:
'''
we need matrix with: (in onehotencode() method it converts 1 row = 1 product, so thts why we use crosstab for encoding 1 row = 1 order)
- rows = orders
- columns = departments
'''
basket_matrix = pd.crosstab(
    order_departments['order_id'],
    order_departments['department']
)

# Convert count values to True/False as fp wants transactional matrix not w int instead it wants true or false hence
basket_matrix = basket_matrix > 0

basket_matrix.head()

department,alcohol,babies,bakery,beverages,breakfast,bulk,canned goods,dairy eggs,deli,dry goods pasta,...,household,international,meat seafood,missing,other,pantry,personal care,pets,produce,snacks
order_id,,,,,,,,,,,,,,,,,,,,,
2,False,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,True,False,False,True,False
3,False,False,True,False,False,False,False,True,False,False,...,False,False,True,False,False,False,False,False,True,False
4,False,False,True,True,True,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,True
5,False,False,False,True,False,False,False,True,True,True,...,True,True,True,False,False,True,True,False,True,True
6,False,False,False,True,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False


In [8]:
# Check basket size
print('Orders:', basket_matrix.shape[0])
print('Departments:', basket_matrix.shape[1])

Orders: 3214874
Departments: 21


In [9]:
frequent_itemsets = fpgrowth(
    basket_matrix,
    min_support=0.02,  #means the combination must appear in at least 2% of orders
    use_colnames=True
)

# Add length of itemset so it is easier to filter later
frequent_itemsets['itemset_length'] = frequent_itemsets['itemsets'].apply(len)

frequent_itemsets.sort_values('support', ascending=False).head(20) #sort by most frequent combinations and keep 20 most freq combos

'''
Example output:

support	itemsets
0.42	{bananas}
0.38	{milk}
0.31	{milk, bread}
'''

,support,itemsets,itemset_length
0,0.749429,(produce),1
1,0.677270,(dairy eggs),1
17,0.550483,"(produce, dairy eggs)",2
5,0.453315,(beverages),1
6,0.432815,(snacks),1
13,0.367361,(frozen),1
2,0.347725,(pantry),1
319,0.337262,"(produce, snacks)",2
315,0.333564,"(beverages, produce)",2
320,0.322548,"(dairy eggs, snacks)",2


In [10]:
'''
meaning: 

- `antecedents`: if basket has this
- `consequents`: then basket may also have this
- `support`: how common the full combination is
- `confidence`: how often consequent appears when antecedent appears
- `lift`: strength of relation; if lift = 3.2, means buying bread makes butter 3.2x more likely 

'''
rules = association_rules(
    frequent_itemsets,
    metric='lift',
    min_threshold=1.0  #lift >1 is positive correlation so it means relation exists thus 1
)

rules = rules.sort_values(
    ['lift', 'confidence'],
    ascending=False
)

rules.head(20)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
15475,"(produce, dry goods pasta, pantry)","(canned goods, dairy eggs, frozen)",0.083256,0.090252,0.022948,0.275637,3.054074,1.0,0.015434,1.255928,0.733649,0.152420,0.203776,0.264953
15490,"(canned goods, dairy eggs, frozen)","(produce, dry goods pasta, pantry)",0.090252,0.083256,0.022948,0.254268,3.054074,1.0,0.015434,1.229322,0.739291,0.152420,0.186544,0.264953
15488,"(dairy eggs, dry goods pasta, pantry)","(canned goods, produce, frozen)",0.080360,0.094684,0.022948,0.285568,3.016026,1.0,0.015340,1.267184,0.726847,0.150881,0.210849,0.263968
15477,"(canned goods, produce, frozen)","(dairy eggs, dry goods pasta, pantry)",0.094684,0.080360,0.022948,0.242368,3.016026,1.0,0.015340,1.213835,0.738347,0.150881,0.176165,0.263968
15359,"(produce, snacks, dry goods pasta)","(canned goods, dairy eggs, frozen)",0.089643,0.090252,0.024357,0.271717,3.010631,1.0,0.016267,1.249167,0.733606,0.156601,0.199467,0.270799
15362,"(canned goods, dairy eggs, frozen)","(produce, snacks, dry goods pasta)",0.090252,0.089643,0.024357,0.269881,3.010631,1.0,0.016267,1.246862,0.734098,0.156601,0.197986,0.270799
15500,"(dry goods pasta, pantry)","(canned goods, produce, dairy eggs, frozen)",0.092687,0.082404,0.022948,0.247590,3.004598,1.0,0.015311,1.219543,0.735332,0.150835,0.180021,0.263038
15465,"(canned goods, produce, dairy eggs, frozen)","(dry goods pasta, pantry)",0.082404,0.092687,0.022948,0.278486,3.004598,1.0,0.015311,1.257513,0.727092,0.150835,0.204780,0.263038
15144,"(produce, dry goods pasta, pantry)","(canned goods, dairy eggs, snacks)",0.083256,0.093563,0.023372,0.280730,3.000448,1.0,0.015583,1.260218,0.727265,0.152316,0.206487,0.265267
15161,"(canned goods, dairy eggs, snacks)","(produce, dry goods pasta, pantry)",0.093563,0.083256,0.023372,0.249804,3.000448,1.0,0.015583,1.222006,0.735535,0.152316,0.181674,0.265267


In [11]:
#raw output gives frozenset so we convert that to readable rules then save
def itemset_to_text(itemset):
    return ', '.join(sorted(list(itemset)))

readable_rules = rules.copy()
readable_rules['if_basket_has'] = readable_rules['antecedents'].apply(itemset_to_text)
readable_rules['then_recommend'] = readable_rules['consequents'].apply(itemset_to_text)

readable_rules = readable_rules[[
    'if_basket_has',
    'then_recommend',
    'support',
    'confidence',
    'lift'
]]

readable_rules.head(20)

,if_basket_has,then_recommend,support,confidence,lift
15475,"dry goods pasta, pantry, produce","canned goods, dairy eggs, frozen",0.022948,0.275637,3.054074
15490,"canned goods, dairy eggs, frozen","dry goods pasta, pantry, produce",0.022948,0.254268,3.054074
15488,"dairy eggs, dry goods pasta, pantry","canned goods, frozen, produce",0.022948,0.285568,3.016026
15477,"canned goods, frozen, produce","dairy eggs, dry goods pasta, pantry",0.022948,0.242368,3.016026
15359,"dry goods pasta, produce, snacks","canned goods, dairy eggs, frozen",0.024357,0.271717,3.010631
15362,"canned goods, dairy eggs, frozen","dry goods pasta, produce, snacks",0.024357,0.269881,3.010631
15500,"dry goods pasta, pantry","canned goods, dairy eggs, frozen, produce",0.022948,0.247590,3.004598
15465,"canned goods, dairy eggs, frozen, produce","dry goods pasta, pantry",0.022948,0.278486,3.004598
15144,"dry goods pasta, pantry, produce","canned goods, dairy eggs, snacks",0.023372,0.280730,3.000448
15161,"canned goods, dairy eggs, snacks","dry goods pasta, pantry, produce",0.023372,0.249804,3.000448


In [12]:
# Keep only useful rules for backend and save them for db and later use 
backend_rules = readable_rules[
    (readable_rules['confidence'] >= 0.2) &
    (readable_rules['lift'] >= 1.0)
].copy()

backend_rules.to_csv('department_association_rules.csv', index=False)

print('Saved department_association_rules.csv')
print('Backend-ready rules:', len(backend_rules))

Saved department_association_rules.csv
Backend-ready rules: 13826


In [13]:
#to check if correct or no 

def recommend_departments(current_departments, top_n=5):
    current_departments = set(current_departments)

    matched_rules = []

    for _, row in rules.iterrows():
        antecedents = set(row['antecedents'])
        consequents = set(row['consequents'])

        # Rule is useful only if basket already has all antecedent departments
        if antecedents.issubset(current_departments):
            for dept in consequents:
                if dept not in current_departments:
                    matched_rules.append({
                        'recommended_department': dept,
                        'confidence': row['confidence'],
                        'lift': row['lift']
                    })

    result = pd.DataFrame(matched_rules)

    if result.empty:
        return result

    result = result.sort_values(['lift', 'confidence'], ascending=False)
    result = result.drop_duplicates(subset=['recommended_department'])

    return result.head(top_n)

In [14]:
# Example check
recommend_departments(['produce'], top_n=5)

,recommended_department,confidence,lift
0,pantry,0.02891,1.269605
1,canned goods,0.02891,1.269605
2,dairy eggs,0.02891,1.269605
3,deli,0.02891,1.269605
4,frozen,0.02891,1.269605
